In [2]:
# ======================================================================
# Standalone ML pipeline for ChEMBL M3 consensus labels
#   - Reads your curated CSV (with smiles + consensus_label)
#   - Builds RDKit Morgan fingerprints (ECFP-like)
#   - Auto sample-weights from consensus_label (active vs active_single etc.)
#   - Scaffold-aware CV (Bemis–Murcko scaffolds) via GroupKFold
#   - Trains a classifier (LogisticRegression by default)
#   - Reports robust metrics (ROC-AUC, PR-AUC, MCC, balanced accuracy)
#
# IMPORTANT NOTE ABOUT pChEMBL:
#   pChEMBL is derived from the same potency values that define your label.
#   Including it as a feature usually causes "label leakage" and inflated scores.
#   Therefore INCLUDE_PCHEMBL_FEATURE is False by default.
# ======================================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import os

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    balanced_accuracy_score,
)
import sys
print(sys.executable)
print(sys.version)

from rdkit import Chem
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit.DataStructs import ConvertToNumpyArray

print("RDKit version:", rdkit.__version__)


os.chdir(r"C:\Users\jdrew\Desktop\M3")

# ----------------------------
# USER SETTINGS
# ----------------------------
DATA_PATH = r"C:\Users\jdrew\Desktop\M3\ChEMBL_M3_consensus_labels_more_negatives_with_meta.csv"

# which labels count as positive/negative
POS_LABELS = {"active", "active_single"}
NEG_LABELS = {"inactive", "inactive_single"}

# sample weights (tune if you want)
WEIGHTS = {
    "active": 1.0,
    "inactive": 1.0,
    "active_single": 0.5,
    "inactive_single": 0.7,
}

# RDKit fingerprint settings
FP_RADIUS = 2
FP_NBITS = 2048
FP_USE_CHIRALITY = True

# Scaffold CV settings
N_SPLITS = 5

# pChEMBL feature (NOT recommended; default off)
INCLUDE_PCHEMBL_FEATURE = False
PCHEMBL_COL = "pChEMBL Value"   # only used if present in your file + INCLUDE_PCHEMBL_FEATURE=True

# Model (baseline, strong and stable)
MODEL = LogisticRegression(
    max_iter=5000,
    solver="liblinear",
    class_weight="balanced",   # helps with imbalance
)

# ----------------------------
# RDKit imports (required)
# ----------------------------
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem
    from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
    from rdkit.Chem.Scaffolds import MurckoScaffold
except Exception as e:
    raise RuntimeError(
        "RDKit is required for this pipeline (SMILES -> fingerprints/scaffolds). "
        "Please install RDKit in your environment."
    ) 
print(GetMorganGenerator)

# ======================================================================
# 1) Load curated data
# ======================================================================
df = pd.read_csv(DATA_PATH)
print("Loaded:", df.shape)
print("Columns:", df.columns.tolist())

needed = {"smiles", "consensus_label"}
missing = needed - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in input CSV: {missing}")

# Keep only ML-usable rows (should already be filtered, but make it safe)
df = df[df["consensus_label"].isin(POS_LABELS | NEG_LABELS)].copy()
print("After label filter:", df.shape)

# Binary target
df["y"] = df["consensus_label"].isin(POS_LABELS).astype(int)

# Sample weights from label type
df["w"] = df["consensus_label"].map(WEIGHTS).astype(float)

# Drop rows with missing SMILES
df = df.dropna(subset=["smiles"]).copy()
df["smiles"] = df["smiles"].astype(str).str.strip()
df = df[df["smiles"] != ""].copy()
print("After SMILES filter:", df.shape)

# Optional: include pChEMBL as a numeric feature (NOT recommended)
use_pchembl = INCLUDE_PCHEMBL_FEATURE and (PCHEMBL_COL in df.columns)
if use_pchembl:
    df["pchembl_feat"] = pd.to_numeric(df[PCHEMBL_COL], errors="coerce")
    # if pChEMBL missing for some rows, we can impute 0 or drop; dropping is cleaner
    df = df.dropna(subset=["pchembl_feat"]).copy()
    print("After pChEMBL availability filter:", df.shape)


# ======================================================================
# 2) Scaffold function (Bemis–Murcko)
# ======================================================================
def smiles_to_scaffold(smiles: str) -> str:
    """Return Bemis–Murcko scaffold SMILES; fallback to empty string if invalid."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return ""
    scaf = MurckoScaffold.GetScaffoldForMol(mol)
    if scaf is None:
        return ""
    return Chem.MolToSmiles(scaf, isomericSmiles=False)

df["scaffold"] = df["smiles"].apply(smiles_to_scaffold)

# Drop invalid scaffolds (invalid SMILES etc.)
bad = (df["scaffold"] == "")
if bad.any():
    print(f"Dropping {bad.sum()} rows with invalid scaffold/SMILES.")
    df = df[~bad].copy()

# Groups for GroupKFold
groups = df["scaffold"].values


# ======================================================================
# 3) Fingerprint transformer (sklearn compatible)
# ======================================================================
class MorganFeaturizer(BaseEstimator, TransformerMixin):
    def __init__(self, radius=2, n_bits=2048, use_chirality=True):
        self.radius = radius
        self.n_bits = n_bits
        self.use_chirality = use_chirality
        self._gen = None

    def fit(self, X, y=None):
        # Initialize generator once
        self._gen = GetMorganGenerator(
            radius=self.radius,
            fpSize=self.n_bits,
            includeChirality=self.use_chirality,
        )
        return self

    def transform(self, X):
        if self._gen is None:
            self.fit(X)

        X = np.asarray(X, dtype=object)
        feats = np.zeros((len(X), self.n_bits), dtype=np.float32)

        for i, smi in enumerate(X):
            mol = Chem.MolFromSmiles(str(smi))
            if mol is None:
                continue
            bv = self._gen.GetFingerprint(mol)
            arr = np.zeros((self.n_bits,), dtype=np.int8)
            ConvertToNumpyArray(bv, arr)
            feats[i, :] = arr

        return feats


# ======================================================================
# 4) Build model pipeline
#     - fingerprints -> (optional scaler) -> classifier
# Note: scaling is not required for bit vectors, but it does not hurt LR.
# ======================================================================
# If you include pChEMBL, we append it as an extra column after fingerprints.
class AppendPchembl(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # X is a tuple: (fp_matrix, pchembl_vector)
        fp, pchembl = X
        pchembl = np.asarray(pchembl).reshape(-1, 1).astype(np.float32)
        return np.hstack([fp, pchembl])

fp_step = ("fp", MorganFeaturizer(radius=FP_RADIUS, n_bits=FP_NBITS, use_chirality=FP_USE_CHIRALITY))

if use_pchembl:
    # We build features manually (still "standalone", just explicit)
    # Pipeline here will take SMILES and later we append pchembl outside the pipeline per fold.
    # (Keeping it simple and readable.)
    pass
else:
    pipe = Pipeline([
        fp_step,
        ("scaler", StandardScaler(with_mean=False)),  # with_mean=False for sparse-like binary matrices
        ("clf", MODEL),
    ])


# ======================================================================
# 5) Scaffold-aware CV evaluation
# ======================================================================
X_smiles = df["smiles"].values
y = df["y"].values.astype(int)
w = df["w"].values.astype(float)

cv = GroupKFold(n_splits=N_SPLITS)

metrics = {
    "roc_auc": [],
    "pr_auc": [],
    "bal_acc": [],
    "mcc": [],
}

for fold, (tr, te) in enumerate(cv.split(X_smiles, y, groups=groups), start=1):
    X_tr, X_te = X_smiles[tr], X_smiles[te]
    y_tr, y_te = y[tr], y[te]
    w_tr = w[tr]

    if use_pchembl:
        # Build fold-specific features with pChEMBL appended (LEAKAGE WARNING applies!)
        fp = MorganFeaturizer(radius=FP_RADIUS, n_bits=FP_NBITS, use_chirality=FP_USE_CHIRALITY)
        Xtr_fp = fp.fit_transform(X_tr)
        Xte_fp = fp.transform(X_te)

        Xtr = np.hstack([Xtr_fp, df["pchembl_feat"].values[tr].reshape(-1, 1)])
        Xte = np.hstack([Xte_fp, df["pchembl_feat"].values[te].reshape(-1, 1)])

        scaler = StandardScaler(with_mean=False)
        Xtr = scaler.fit_transform(Xtr)
        Xte = scaler.transform(Xte)

        clf = LogisticRegression(max_iter=5000, solver="liblinear", class_weight="balanced")
        clf.fit(Xtr, y_tr, sample_weight=w_tr)
        proba = clf.predict_proba(Xte)[:, 1]
        pred = (proba >= 0.5).astype(int)
    else:
        pipe.fit(X_tr, y_tr, clf__sample_weight=w_tr)
        proba = pipe.predict_proba(X_te)[:, 1]
        pred = (proba >= 0.5).astype(int)

    # Metrics
    try:
        roc = roc_auc_score(y_te, proba)
    except Exception:
        roc = np.nan
    pr = average_precision_score(y_te, proba)
    bal = balanced_accuracy_score(y_te, pred)
    mcc = matthews_corrcoef(y_te, pred)

    metrics["roc_auc"].append(roc)
    metrics["pr_auc"].append(pr)
    metrics["bal_acc"].append(bal)
    metrics["mcc"].append(mcc)

    print(
        f"[Fold {fold}] "
        f"ROC-AUC={roc:.3f} | PR-AUC={pr:.3f} | BalAcc={bal:.3f} | MCC={mcc:.3f} "
        f"| n_test={len(te)} | pos_test={int(y_te.sum())}"
    )

print("\n=== Scaffold CV summary ===")
for k, vals in metrics.items():
    vals = np.array(vals, dtype=float)
    print(f"{k}: mean={np.nanmean(vals):.3f}  std={np.nanstd(vals):.3f}  folds={len(vals)}")


# ======================================================================
# 6) Optional: train final model on all data and save it
# (Uncomment if you want a final fitted model object)
# ======================================================================
# if not use_pchembl:
#     pipe.fit(X_smiles, y, clf__sample_weight=w)
#     import joblib
#     joblib.dump(pipe, r"C:\Users\jdrew\Desktop\m3_model_scaffoldcv.joblib")
#     print("[SAVED MODEL] m3_model_scaffoldcv.joblib")


c:\Users\jdrew\Anaconda3\envs\openms_env\python.exe
3.10.16 | packaged by conda-forge | (main, Dec  5 2024, 14:07:43) [MSC v.1942 64 bit (AMD64)]
RDKit version: 2025.09.5
Loaded: (2268, 13)
Columns: ['molecule_chembl_id', 'smiles', 'molecular_weight', 'molecule_name', 'n', 'p_median', 'p_median_kiki', 'frac_active', 'frac_inactive', 'n_strong_active', 'n_strong_inactive', 'any_active_vote', 'consensus_label']
After label filter: (2268, 13)
After SMILES filter: (2268, 15)
[Fold 1] ROC-AUC=0.974 | PR-AUC=0.992 | BalAcc=0.902 | MCC=0.830 | n_test=454 | pos_test=354
[Fold 2] ROC-AUC=0.960 | PR-AUC=0.988 | BalAcc=0.867 | MCC=0.799 | n_test=454 | pos_test=363
[Fold 3] ROC-AUC=0.834 | PR-AUC=0.939 | BalAcc=0.796 | MCC=0.639 | n_test=454 | pos_test=369
[Fold 4] ROC-AUC=0.968 | PR-AUC=0.991 | BalAcc=0.900 | MCC=0.821 | n_test=453 | pos_test=360
[Fold 5] ROC-AUC=0.952 | PR-AUC=0.975 | BalAcc=0.879 | MCC=0.748 | n_test=453 | pos_test=342

=== Scaffold CV summary ===
roc_auc: mean=0.938  std=0.052

In [4]:
# ======================================================================
# Train + Scaffold-CV benchmark for ChEMBL M3 (consensus labels)
#   - Morgan fingerprints (RDKit MorganGenerator)
#   - Optional physchem descriptors (RDKit)
#   - Sample weights from consensus_label
#   - Scaffold-aware CV (Bemis–Murcko, GroupKFold)
#   - Compare:
#       A) FP-only
#       B) FP + physchem
#   - Save final chosen model (joblib)
# ======================================================================

import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    balanced_accuracy_score,
)

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit.DataStructs import ConvertToNumpyArray
from rdkit.Chem import Descriptors

# ----------------------------
# USER SETTINGS
# ----------------------------
DATA_PATH = r"C:\Users\jdrew\Desktop\M3\ChEMBL_M3_consensus_labels_more_negatives_with_meta.csv"

POS_LABELS = {"active", "active_single"}
NEG_LABELS = {"inactive", "inactive_single"}

WEIGHTS = {
    "active": 1.0,
    "inactive": 1.0,
    "active_single": 0.5,
    "inactive_single": 0.7,
}

# Fingerprints
FP_RADIUS = 2
FP_NBITS = 2048
FP_USE_CHIRALITY = True

# Scaffold CV
N_SPLITS = 5
RANDOM_STATE = 42  # only used for reproducibility in any randomized steps (none here)

# Classifier baseline
def make_clf():
    return LogisticRegression(
        max_iter=5000,
        solver="liblinear",
        class_weight="balanced",
    )

# Output model path
SAVE_MODEL_PATH = r"C:\Users\jdrew\Desktop\M3\m3_fp_physchem_scaffoldcv.joblib"


# ======================================================================
# 1) Load and prepare data
# ======================================================================
df = pd.read_csv(DATA_PATH)
need = {"smiles", "consensus_label"}
missing = need - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[df["consensus_label"].isin(POS_LABELS | NEG_LABELS)].copy()
df = df.dropna(subset=["smiles"]).copy()
df["smiles"] = df["smiles"].astype(str).str.strip()
df = df[df["smiles"] != ""].copy()

df["y"] = df["consensus_label"].isin(POS_LABELS).astype(int)
df["w"] = df["consensus_label"].map(WEIGHTS).astype(float)

print("Loaded & filtered:", df.shape)
print(df["consensus_label"].value_counts())

# ======================================================================
# 2) Scaffold groups
# ======================================================================
def smiles_to_scaffold(smiles: str) -> str:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return ""
    scaf = MurckoScaffold.GetScaffoldForMol(mol)
    if scaf is None:
        return ""
    return Chem.MolToSmiles(scaf, isomericSmiles=False)

df["scaffold"] = df["smiles"].apply(smiles_to_scaffold)
bad = df["scaffold"].eq("")
if bad.any():
    print(f"Dropping {bad.sum()} rows with invalid SMILES/scaffold.")
    df = df[~bad].copy()

X_smiles = df["smiles"].values
y = df["y"].values.astype(int)
w = df["w"].values.astype(float)
groups = df["scaffold"].values

cv = GroupKFold(n_splits=N_SPLITS)


# ======================================================================
# 3) Feature transformers
# ======================================================================

class MorganFeaturizer(BaseEstimator, TransformerMixin):
    def __init__(self, radius=2, n_bits=2048, use_chirality=True):
        self.radius = radius
        self.n_bits = n_bits
        self.use_chirality = use_chirality
        self._gen = None  # not picklable -> rebuild on demand

    def _build_gen(self):
        self._gen = GetMorganGenerator(
            radius=self.radius,
            fpSize=self.n_bits,
            includeChirality=self.use_chirality,
        )

    def fit(self, X, y=None):
        self._build_gen()
        return self

    def transform(self, X):
        if self._gen is None:
            self._build_gen()

        X = np.asarray(X, dtype=object)
        feats = np.zeros((len(X), self.n_bits), dtype=np.float32)

        for i, smi in enumerate(X):
            mol = Chem.MolFromSmiles(str(smi))
            if mol is None:
                continue
            bv = self._gen.GetFingerprint(mol)
            arr = np.zeros((self.n_bits,), dtype=np.int8)
            ConvertToNumpyArray(bv, arr)
            feats[i, :] = arr

        return feats

    # ---- KEY PART: make object picklable by dropping _gen ----
    def __getstate__(self):
        state = self.__dict__.copy()
        state["_gen"] = None
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)
        # _gen stays None until first transform/fit, then rebuilt

class PhysChemFeaturizer(BaseEstimator, TransformerMixin):
    """
    Compact, interpretable physchem feature set.
    Returns 6 columns:
      MolWt, MolLogP, HBD, HBA, TPSA, NumRotatableBonds
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=object)
        feats = np.zeros((len(X), 6), dtype=np.float32)

        for i, smi in enumerate(X):
            mol = Chem.MolFromSmiles(str(smi))
            if mol is None:
                feats[i, :] = np.nan
                continue
            feats[i, 0] = Descriptors.MolWt(mol)
            feats[i, 1] = Descriptors.MolLogP(mol)
            feats[i, 2] = Descriptors.NumHDonors(mol)
            feats[i, 3] = Descriptors.NumHAcceptors(mol)
            feats[i, 4] = Descriptors.TPSA(mol)
            feats[i, 5] = Descriptors.NumRotatableBonds(mol)

        # Replace NaNs (rare if SMILES valid) with column medians
        if np.isnan(feats).any():
            col_median = np.nanmedian(feats, axis=0)
            inds = np.where(np.isnan(feats))
            feats[inds] = np.take(col_median, inds[1])
        return feats


class ConcatFeatures(BaseEstimator, TransformerMixin):
    """Concatenate outputs of two transformers applied to the same input X."""
    def __init__(self, t1, t2):
        self.t1 = t1
        self.t2 = t2

    def fit(self, X, y=None):
        self.t1.fit(X, y)
        self.t2.fit(X, y)
        return self

    def transform(self, X):
        A = self.t1.transform(X)
        B = self.t2.transform(X)
        return np.hstack([A, B])


# ======================================================================
# 4) Pipelines to compare
# ======================================================================
pipe_fp_only = Pipeline([
    ("feat", MorganFeaturizer(radius=FP_RADIUS, n_bits=FP_NBITS, use_chirality=FP_USE_CHIRALITY)),
    ("scaler", StandardScaler(with_mean=False)),
    ("clf", make_clf()),
])

pipe_fp_physchem = Pipeline([
    ("feat", ConcatFeatures(
        MorganFeaturizer(radius=FP_RADIUS, n_bits=FP_NBITS, use_chirality=FP_USE_CHIRALITY),
        PhysChemFeaturizer()
    )),
    ("scaler", StandardScaler(with_mean=False)),
    ("clf", make_clf()),
])


# ======================================================================
# 5) Scaffold CV evaluation function
# ======================================================================
def eval_pipe(pipe, name):
    metrics = {"roc_auc": [], "pr_auc": [], "bal_acc": [], "mcc": []}
    print(f"\n=== {name} ===")

    for fold, (tr, te) in enumerate(cv.split(X_smiles, y, groups=groups), start=1):
        X_tr, X_te = X_smiles[tr], X_smiles[te]
        y_tr, y_te = y[tr], y[te]
        w_tr = w[tr]

        pipe.fit(X_tr, y_tr, clf__sample_weight=w_tr)
        proba = pipe.predict_proba(X_te)[:, 1]
        pred = (proba >= 0.5).astype(int)

        roc = roc_auc_score(y_te, proba)
        pr = average_precision_score(y_te, proba)
        bal = balanced_accuracy_score(y_te, pred)
        mcc = matthews_corrcoef(y_te, pred)

        metrics["roc_auc"].append(roc)
        metrics["pr_auc"].append(pr)
        metrics["bal_acc"].append(bal)
        metrics["mcc"].append(mcc)

        print(
            f"[Fold {fold}] ROC-AUC={roc:.3f} | PR-AUC={pr:.3f} | "
            f"BalAcc={bal:.3f} | MCC={mcc:.3f} | n_test={len(te)} | pos_test={int(y_te.sum())}"
        )

    print(f"\n--- {name} summary ---")
    for k, vals in metrics.items():
        vals = np.array(vals, dtype=float)
        print(f"{k}: mean={vals.mean():.3f}  std={vals.std():.3f}")
    return metrics


# ======================================================================
# 6) Run comparison
# ======================================================================
m1 = eval_pipe(pipe_fp_only, "A) FP-only")
m2 = eval_pipe(pipe_fp_physchem, "B) FP + physchem")


# ======================================================================
# 7) Fit final model on all data (choose FP+physchem by default) + save
# ======================================================================
final_pipe = pipe_fp_physchem  # change to pipe_fp_only if it performs better
final_pipe.fit(X_smiles, y, clf__sample_weight=w)

joblib.dump(final_pipe, SAVE_MODEL_PATH)
print("\n[SAVED MODEL]", SAVE_MODEL_PATH)


Loaded & filtered: (2268, 15)
consensus_label
active_single      1502
inactive_single     463
active              286
inactive             17
Name: count, dtype: int64

=== A) FP-only ===
[Fold 1] ROC-AUC=0.974 | PR-AUC=0.992 | BalAcc=0.902 | MCC=0.830 | n_test=454 | pos_test=354
[Fold 2] ROC-AUC=0.960 | PR-AUC=0.988 | BalAcc=0.867 | MCC=0.799 | n_test=454 | pos_test=363
[Fold 3] ROC-AUC=0.834 | PR-AUC=0.939 | BalAcc=0.796 | MCC=0.639 | n_test=454 | pos_test=369
[Fold 4] ROC-AUC=0.968 | PR-AUC=0.991 | BalAcc=0.900 | MCC=0.821 | n_test=453 | pos_test=360
[Fold 5] ROC-AUC=0.952 | PR-AUC=0.975 | BalAcc=0.879 | MCC=0.748 | n_test=453 | pos_test=342

--- A) FP-only summary ---
roc_auc: mean=0.938  std=0.052
pr_auc: mean=0.977  std=0.020
bal_acc: mean=0.869  std=0.039
mcc: mean=0.767  std=0.070

=== B) FP + physchem ===
[Fold 1] ROC-AUC=0.976 | PR-AUC=0.993 | BalAcc=0.907 | MCC=0.836 | n_test=454 | pos_test=354
[Fold 2] ROC-AUC=0.961 | PR-AUC=0.989 | BalAcc=0.867 | MCC=0.799 | n_test=454 | p

**What this analysis will give you**
- Per-fold misclassification table
- SMILES
- consensus label
- predicted probability
- fase positive / false negative flag
- scaffold
- fold index

**Per-scaffold performance summary**
- number of compounds per scaffold
- MCC / Balanced Accuracy per scaffold
- which scaffolds are systematically hard

**Top “problem scaffolds” CSV**
- exactly where chemistry, not ML, is limiting

In [7]:
# ======================================================================
# Scaffold-aware error analysis for the trained pipeline
# ======================================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.metrics import matthews_corrcoef, balanced_accuracy_score

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold


# ----------------------------
# SETTINGS (must match training)
# ----------------------------
DATA_PATH = r"C:\Users\jdrew\Desktop\M3\ChEMBL_M3_consensus_labels_more_negatives_with_meta.csv"
MODEL_PATH = r"C:\Users\jdrew\Desktop\M3\m3_fp_physchem_scaffoldcv.joblib"

POS_LABELS = {"active", "active_single"}
NEG_LABELS = {"inactive", "inactive_single"}
N_SPLITS = 5

# ----------------------------
# Load data and model
# ----------------------------
df = pd.read_csv(DATA_PATH)
pipe = joblib.load(MODEL_PATH)

df = df[df["consensus_label"].isin(POS_LABELS | NEG_LABELS)].copy()
df = df.dropna(subset=["smiles"]).copy()

df["y"] = df["consensus_label"].isin(POS_LABELS).astype(int)


# ----------------------------
# Scaffold assignment
# ----------------------------
def smiles_to_scaffold(smiles: str) -> str:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return ""
    scaf = MurckoScaffold.GetScaffoldForMol(mol)
    if scaf is None:
        return ""
    return Chem.MolToSmiles(scaf, isomericSmiles=False)

df["scaffold"] = df["smiles"].apply(smiles_to_scaffold)
df = df[df["scaffold"] != ""].copy()

X = df["smiles"].values
y = df["y"].values
groups = df["scaffold"].values

cv = GroupKFold(n_splits=N_SPLITS)


# ======================================================================
# 1) Collect misclassifications per fold
# ======================================================================
rows = []

for fold, (tr, te) in enumerate(cv.split(X, y, groups=groups), start=1):
    X_te = X[te]
    y_te = y[te]

    proba = pipe.predict_proba(X_te)[:, 1]
    pred = (proba >= 0.5).astype(int)

    for i, idx in enumerate(te):
        rows.append({
            "fold": fold,
            "molecule_chembl_id": df.iloc[idx]["molecule_chembl_id"],
            "smiles": df.iloc[idx]["smiles"],
            "scaffold": df.iloc[idx]["scaffold"],
            "true_label": y_te[i],
            "pred_label": pred[i],
            "p_active": proba[i],
            "error_type": (
                "FP" if (pred[i] == 1 and y_te[i] == 0)
                else "FN" if (pred[i] == 0 and y_te[i] == 1)
                else "OK"
            ),
        })

err_df = pd.DataFrame(rows)

err_df.to_csv(
    r"C:\Users\jdrew\Desktop\M3\scaffold_cv_misclassifications.csv",
    index=False,
)
print("[SAVED] scaffold_cv_misclassifications.csv")
print(err_df["error_type"].value_counts())


# ======================================================================
# 2) Per-scaffold performance
# ======================================================================
scaf_stats = []

for scaf, g in err_df.groupby("scaffold"):
    y_true = g["true_label"].values
    y_pred = g["pred_label"].values

    if len(np.unique(y_true)) < 2:
        continue  # MCC undefined

    scaf_stats.append({
        "scaffold": scaf,
        "n": len(g),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "bal_acc": balanced_accuracy_score(y_true, y_pred),
        "n_errors": int((g["error_type"] != "OK").sum()),
    })

scaf_df = pd.DataFrame(scaf_stats).sort_values(
    ["mcc", "n"], ascending=[True, False]
)

scaf_df.to_csv(
    r"C:\Users\jdrew\Desktop\M3\scaffold_performance_summary.csv",
    index=False,
)
print("[SAVED] scaffold_performance_summary.csv")
print(scaf_df.head(10))


# ======================================================================
# 3) Identify "hard scaffolds"
# ======================================================================
hard_scaffolds = scaf_df.query("n >= 10 and mcc < 0.5")

hard_scaffolds.to_csv(
    r"C:\Users\jdrew\Desktop\M3\hard_scaffolds.csv",
    index=False,
)

print("\nHard scaffolds (n>=10 & MCC<0.5):")
print(hard_scaffolds.head(10))


[SAVED] scaffold_cv_misclassifications.csv
error_type
OK    2268
Name: count, dtype: int64
[SAVED] scaffold_performance_summary.csv
                                             scaffold   n  mcc  bal_acc  \
11          c1ccc(C(CC2CC3CCC(C2)[NH2+]3)c2ccccc2)cc1  29  1.0      1.0   
12             c1ccc2c(c1)CCN1Cc3c(ccc4[nH]ccc34)OC21  18  1.0      1.0   
5   O=C(Nc1ccccc1)NC(Cc1ccccc1)C(=O)NC1CCN(Cc2cccc...  17  1.0      1.0   
16                                           c1ccccc1  12  1.0      1.0   
4   O=C(NCc1cccc(-c2cccc(CN3CCNCC3)c2)c1)c1ccc2c(c...  10  1.0      1.0   
0                           C1=C(Cc2ccccn2)c2ccccc2C1   9  1.0      1.0   
1                                 C1=C(c2ccccc2)CCNC1   7  1.0      1.0   
10            O=C1Nc2ccccc2N(C(=O)CN2CCCCC2)c2ccccc21   7  1.0      1.0   
8                 O=C(OCCc1ccccc1)C1=C(c2ccccc2)CCNC1   6  1.0      1.0   
15                             c1ccc2c(c1)Nc1ccccc1S2   5  1.0      1.0   

    n_errors  
11         0  
12         0